# Safe Pilot App Enhancement
## Scenario 1 — Weather Alerts for Parked / Static Vehicles

---

### What this notebook does

When a driver's car is **parked or static** and **severe weather is approaching**, the Safe Pilot app should proactively alert the driver so they can take protective action (move the car to a shelter, evacuate the area, etc.).

### Alert decision rules
An alert is sent **only when all three conditions are true**:

| Condition | Rule |
|---|---|
| Vehicle state | `speed == 0` **OR** `status` is `"parked"` / `"idle"` |
| Weather severity | At least `moderate` (moderate → severe → extreme) |
| Proximity | Weather event is within **30 km** of the vehicle |

### No external dependencies
This notebook uses only the Python standard library — no `pip install` required. ✅

---
## Cell 1 — Imports

`datetime` and `timedelta` are used to calculate the projected arrival time of the weather event.  
`Optional` provides type hints for weather fields that may or may not be present (e.g., hail size, wind speed).

In [ ]:
from datetime import datetime, timedelta
from typing import Optional

print("✅ Imports loaded successfully.")

---
## Cell 2 — Data Structures

Two factory functions simulate the data the Safe Pilot app would receive in production:

### `make_vehicle(...)`
Represents the **current state of the driver's vehicle**:
- `vehicle_id` — registration / identifier
- `speed_kmh` — current speed (`0` = parked/static)
- `status` — `"parked"` | `"idle"` | `"moving"`
- `location` — lat/lon + human-readable address

### `make_weather_event(...)`
Represents an **incoming weather event** near the vehicle:
- `event_type` — type of weather (`"hail"`, `"tornado"`, `"flash_flood"`, `"blizzard"`, `"high_winds"`, `"thunderstorm"`, `"dust_storm"`, `"clear"`)
- `severity` — `"none"` | `"moderate"` | `"severe"` | `"extreme"`
- `distance_km` — how far the event currently is from the vehicle
- `eta_minutes` — estimated minutes until it arrives
- Optional fields: `hail_size_cm`, `rainfall_mm_hr`, `wind_speed_kmh` — enrich the alert message

In [ ]:
def make_vehicle(vehicle_id: str, speed_kmh: float, status: str, location: dict) -> dict:
    """Build a vehicle state dictionary."""
    return {
        "vehicle_id": vehicle_id,
        "speed_kmh": speed_kmh,
        "status": status,
        "location": location,
        "timestamp": datetime.now().isoformat(),
    }


def make_weather_event(
    event_type: str,
    severity: str,
    distance_km: float,
    eta_minutes: int,
    description: str,
    wind_speed_kmh: Optional[float] = None,
    rainfall_mm_hr: Optional[float] = None,
    hail_size_cm: Optional[float] = None,
) -> dict:
    """Build a weather event dictionary."""
    return {
        "event_type": event_type,
        "severity": severity,
        "distance_km": distance_km,
        "eta_minutes": eta_minutes,
        "description": description,
        "wind_speed_kmh": wind_speed_kmh,
        "rainfall_mm_hr": rainfall_mm_hr,
        "hail_size_cm": hail_size_cm,
        "observed_at": datetime.now().isoformat(),
        # Projected arrival time = now + ETA
        "projected_arrival": (
            datetime.now() + timedelta(minutes=eta_minutes)
        ).strftime("%H:%M"),
    }


print("✅ Data structure functions defined.")

---
## Cell 3 — Configuration & Alert Templates

### Severity ranking
`SEVERITY_RANK` maps severity labels to numeric values so they can be compared (`none=0` < `moderate=1` < `severe=2` < `extreme=3`).

### Thresholds (tunable)
| Parameter | Default | Meaning |
|---|---|---|
| `ALERT_SEVERITY_THRESHOLD` | `"moderate"` | Minimum severity to trigger an alert |
| `ALERT_RADIUS_KM` | `30` | Maximum distance (km) for an alert to fire |

### Alert templates
Each weather type has a dedicated message template with `{placeholders}` for dynamic values like arrival time, hail size, wind speed, etc.  
A `"default"` fallback handles any event type not explicitly listed.

In [ ]:
# --- Severity ladder (lowest → highest) ---
SEVERITY_RANK = {"none": 0, "moderate": 1, "severe": 2, "extreme": 3}

# --- Alert thresholds (adjust here to tune sensitivity) ---
ALERT_SEVERITY_THRESHOLD = "moderate"   # alerts fire at moderate and above
ALERT_RADIUS_KM          = 30           # only alert if event is within 30 km

# --- Per-event-type alert message templates ---
ALERT_TEMPLATES = {
    "hail": (
        "Hail storm approaching! Hailstones {hail_detail}are expected to reach your area "
        "around {arrival}. Move your vehicle to a covered shelter (garage, carport, or "
        "underpass) to prevent damage."
    ),
    "thunderstorm": (
        "Severe thunderstorm approaching your area around {arrival}. "
        "Ensure your vehicle is parked away from tall trees and power lines. "
        "Stay indoors until the storm passes."
    ),
    "tornado": (
        "TORNADO WARNING! A tornado is {distance:.1f} km away and may reach you around {arrival}. "
        "Do NOT shelter in your vehicle. Move to a sturdy building or underground shelter immediately!"
    ),
    "flash_flood": (
        "Flash flood warning in effect. Heavy rainfall {rain_detail}could cause flooding near your "
        "parked location around {arrival}. Move your vehicle to higher ground immediately."
    ),
    "blizzard": (
        "Blizzard conditions approaching around {arrival}. Expect heavy snowfall and whiteout "
        "conditions. Move your vehicle to a covered area and avoid travel until conditions improve."
    ),
    "high_winds": (
        "Severe wind advisory: Gusts up to {wind_detail}expected around {arrival}. "
        "Park your vehicle away from trees, billboards, and unstable structures to avoid damage."
    ),
    "dust_storm": (
        "Dust storm (haboob) approaching around {arrival}. Visibility may drop to near zero. "
        "Keep your vehicle parked and stay inside until the storm clears."
    ),
    # Fallback for any weather type not listed above
    "default": (
        "Severe weather ({event_type}) is approaching your parked vehicle and expected around {arrival}. "
        "Take precautions to ensure your vehicle is in a safe location."
    ),
}

print("✅ Configuration and alert templates loaded.")

---
## Cell 4 — Core Alert Logic

Three focused functions that implement the decision pipeline:

### `is_vehicle_parked(vehicle)`
Returns `True` if the vehicle is parked **or** stationary.  
Catches both explicit `status="parked"`/`"idle"` and the case where speed is `0` but status wasn't updated.

### `is_alert_required(weather)`
Returns `True` only if **all** of the following hold:
- Weather event is not `"clear"`
- Severity ≥ `ALERT_SEVERITY_THRESHOLD`
- Distance ≤ `ALERT_RADIUS_KM`

### `build_alert_message(vehicle, weather)`
Selects the right template for the event type and fills in dynamic values (arrival time, hail size, wind speed, rainfall rate).  
Optional fields are only included in the message if they were provided in the weather event.

### `evaluate_and_alert(vehicle, weather)` — Main entry point
Orchestrates the checks and returns a result dict:
```python
{
    "alert_triggered": bool,
    "alert_message":   str | None,
    "reason":          str   # explains the decision
}
```

In [ ]:
def is_vehicle_parked(vehicle: dict) -> bool:
    """Return True if the vehicle is parked or completely stationary."""
    return vehicle["status"] in ("parked", "idle") or vehicle["speed_kmh"] == 0


def is_alert_required(weather: dict) -> bool:
    """Return True if the weather event is severe enough and close enough to alert."""
    severity_ok = (
        SEVERITY_RANK.get(weather["severity"], 0)
        >= SEVERITY_RANK[ALERT_SEVERITY_THRESHOLD]
    )
    distance_ok = weather["distance_km"] <= ALERT_RADIUS_KM
    not_clear   = weather["event_type"] != "clear"
    return severity_ok and distance_ok and not_clear


def build_alert_message(vehicle: dict, weather: dict) -> str:
    """Compose the driver-facing alert message using the appropriate template."""
    template = ALERT_TEMPLATES.get(weather["event_type"], ALERT_TEMPLATES["default"])

    # Build optional detail sub-strings (empty string if data not available)
    hail_detail = f"(up to {weather['hail_size_cm']} cm) "  if weather.get("hail_size_cm")    else ""
    rain_detail = f"({weather['rainfall_mm_hr']} mm/hr) "   if weather.get("rainfall_mm_hr")   else ""
    wind_detail = f"{weather['wind_speed_kmh']} km/h "      if weather.get("wind_speed_kmh")   else ""

    return template.format(
        arrival    = weather["projected_arrival"],
        distance   = weather["distance_km"],
        hail_detail= hail_detail,
        rain_detail= rain_detail,
        wind_detail= wind_detail,
        event_type = weather["event_type"].replace("_", " ").title(),
    )


def evaluate_and_alert(vehicle: dict, weather: dict) -> dict:
    """
    Main entry point — checks vehicle state and weather conditions,
    then decides whether to send an alert.
    """
    parked       = is_vehicle_parked(vehicle)
    alert_needed = is_alert_required(weather)

    # Gate 1: vehicle must be parked for Scenario 1
    if not parked:
        return {
            "alert_triggered": False,
            "alert_message":   None,
            "reason": (
                f"Vehicle is moving ({vehicle['speed_kmh']} km/h). "
                "Scenario 1 alerts only apply to parked/static vehicles."
            ),
        }

    # Gate 2: weather must meet severity + proximity thresholds
    if not alert_needed:
        sev, dist, etype = weather["severity"], weather["distance_km"], weather["event_type"]
        if etype == "clear":
            reason = "Weather conditions are clear. No alert needed."
        elif SEVERITY_RANK.get(sev, 0) < SEVERITY_RANK[ALERT_SEVERITY_THRESHOLD]:
            reason = (
                f"Weather event '{etype}' has severity '{sev}', which is below "
                f"the alert threshold ('{ALERT_SEVERITY_THRESHOLD}'). No alert sent."
            )
        else:
            reason = (
                f"Weather event '{etype}' is {dist:.1f} km away, which is beyond "
                f"the alert radius ({ALERT_RADIUS_KM} km). No alert sent."
            )
        return {"alert_triggered": False, "alert_message": None, "reason": reason}

    # Both gates passed — generate the alert
    return {
        "alert_triggered": True,
        "alert_message":   build_alert_message(vehicle, weather),
        "reason": (
            f"Vehicle is parked. '{weather['event_type']}' ({weather['severity']}) "
            f"is {weather['distance_km']:.1f} km away, ETA {weather['eta_minutes']} min."
        ),
    }


print("✅ Core alert logic defined.")

---
## Cell 5 — Display Helper

`print_result(...)` formats the scenario output into a readable block showing:
- Vehicle details (ID, status, speed, location)
- Weather details (type, severity, distance, ETA)
- Decision outcome (`ALERT TRIGGERED` or `NO ALERT`)
- Reason for the decision
- Full alert message (only when alert is triggered)

In [ ]:
def print_result(scenario_name: str, vehicle: dict, weather: dict, result: dict):
    """Pretty-print the evaluation result for a scenario."""
    sep = "=" * 70
    print(f"\n{sep}")
    print(f"  SCENARIO: {scenario_name}")
    print(sep)
    print(f"  Vehicle  : {vehicle['vehicle_id']}  |  "
          f"Status: {vehicle['status']}  |  Speed: {vehicle['speed_kmh']} km/h")
    print(f"  Location : {vehicle['location']['address']}")
    print(f"  Weather  : {weather['event_type'].replace('_',' ').title()}  |  "
          f"Severity: {weather['severity'].upper()}  |  "
          f"Distance: {weather['distance_km']} km  |  ETA: {weather['eta_minutes']} min")
    print("-" * 70)

    status_label = "🚨 ALERT TRIGGERED" if result["alert_triggered"] else "✅ NO ALERT"
    print(f"  [{status_label}]")
    print(f"  Reason : {result['reason']}")

    if result["alert_triggered"]:
        print(f"\n  📢 MESSAGE:\n  {result['alert_message']}")
    print(sep)


print("✅ Display helper defined.")

---
## Cell 6 — Sample Scenarios (Input Data)

Ten test cases cover all key decision paths:

### Positive scenarios — alert SHOULD fire
| # | Vehicle | Weather event | Why alert fires |
|---|---|---|---|
| P1 | Parked, Chennai | Hail — Severe, 15 km | Parked + severe + within radius |
| P2 | Idle (speed=0), Bengaluru | Tornado — Extreme, 8 km | Speed=0 counts as parked |
| P3 | Parked, Pune | Flash Flood — Severe, 5 km | Parked + severe + very close |
| P4 | Parked, Manali | Blizzard — Extreme, 20 km | Parked + extreme + within radius |
| P5 | Parked, Ahmedabad | High Winds — Moderate, 10 km | Moderate severity still triggers |

### Negative scenarios — alert should NOT fire
| # | Vehicle | Weather event | Why alert is suppressed |
|---|---|---|---|
| N1 | Moving 60 km/h, Delhi | Hail — Severe, 10 km | Car is moving (Scenario 1 skips) |
| N2 | Parked, Coimbatore | Thunderstorm — None, 5 km | Severity below threshold |
| N3 | Parked, Hyderabad | Hail — Severe, 80 km | Beyond 30 km alert radius |
| N4 | Parked, Kochi | Clear — None, 0 km | No weather event |
| N5 | Moving 85 km/h, Jaipur | Clear — None, 0 km | Moving + no event (double negative) |

In [ ]:
# ============================================================
# POSITIVE SCENARIOS — alert should fire
# ============================================================

positive_scenarios = [
    (
        "Positive 1 – Parked car + Severe Hail Storm approaching",
        make_vehicle(
            vehicle_id="TN-01-AB-1234", speed_kmh=0, status="parked",
            location={"lat": 13.0827, "lon": 80.2707, "address": "Anna Nagar, Chennai"},
        ),
        make_weather_event(
            event_type="hail", severity="severe",
            distance_km=15, eta_minutes=25,
            description="Large hailstones expected",
            hail_size_cm=3.5,
        ),
    ),
    (
        "Positive 2 – Static car (speed=0, status=idle) + Tornado Warning",
        make_vehicle(
            vehicle_id="KA-05-CD-5678", speed_kmh=0, status="idle",
            location={"lat": 12.9716, "lon": 77.5946, "address": "Indiranagar, Bengaluru"},
        ),
        make_weather_event(
            event_type="tornado", severity="extreme",
            distance_km=8, eta_minutes=12,
            description="EF3 tornado on ground",
        ),
    ),
    (
        "Positive 3 – Parked car + Flash Flood (heavy rainfall)",
        make_vehicle(
            vehicle_id="MH-12-EF-9012", speed_kmh=0, status="parked",
            location={"lat": 18.5204, "lon": 73.8567, "address": "Koregaon Park, Pune"},
        ),
        make_weather_event(
            event_type="flash_flood", severity="severe",
            distance_km=5, eta_minutes=20,
            description="Extreme rainfall causing rapid flooding",
            rainfall_mm_hr=120,
        ),
    ),
    (
        "Positive 4 – Parked car + Blizzard approaching",
        make_vehicle(
            vehicle_id="HP-65-GH-3456", speed_kmh=0, status="parked",
            location={"lat": 32.2190, "lon": 77.1910, "address": "Mall Road, Manali"},
        ),
        make_weather_event(
            event_type="blizzard", severity="extreme",
            distance_km=20, eta_minutes=45,
            description="Heavy snowfall with near-zero visibility",
        ),
    ),
    (
        "Positive 5 – Parked car + Severe High Winds",
        make_vehicle(
            vehicle_id="GJ-01-IJ-7890", speed_kmh=0, status="parked",
            location={"lat": 23.0225, "lon": 72.5714, "address": "Satellite, Ahmedabad"},
        ),
        make_weather_event(
            event_type="high_winds", severity="moderate",
            distance_km=10, eta_minutes=30,
            description="Damaging wind gusts",
            wind_speed_kmh=95,
        ),
    ),
]

# ============================================================
# NEGATIVE SCENARIOS — alert should NOT fire
# ============================================================

negative_scenarios = [
    (
        "Negative 1 – Car is MOVING (not parked), severe weather present",
        make_vehicle(
            vehicle_id="DL-08-KL-2345", speed_kmh=60, status="moving",
            location={"lat": 28.6139, "lon": 77.2090, "address": "Connaught Place, Delhi"},
        ),
        make_weather_event(
            event_type="hail", severity="severe",
            distance_km=10, eta_minutes=15,
            description="Large hail approaching",
            hail_size_cm=2.0,
        ),
    ),
    (
        "Negative 2 – Parked car + Weather severity BELOW threshold (minor drizzle)",
        make_vehicle(
            vehicle_id="TN-10-MN-6789", speed_kmh=0, status="parked",
            location={"lat": 11.0168, "lon": 76.9558, "address": "RS Puram, Coimbatore"},
        ),
        make_weather_event(
            event_type="thunderstorm", severity="none",  # below 'moderate' threshold
            distance_km=5, eta_minutes=40,
            description="Light drizzle with distant thunder",
        ),
    ),
    (
        "Negative 3 – Parked car + Weather event is FAR AWAY (beyond radius)",
        make_vehicle(
            vehicle_id="AP-28-OP-1357", speed_kmh=0, status="parked",
            location={"lat": 17.3850, "lon": 78.4867, "address": "Jubilee Hills, Hyderabad"},
        ),
        make_weather_event(
            event_type="hail", severity="severe",
            distance_km=80,              # beyond 30 km alert radius
            eta_minutes=120,
            description="Hail storm far from current location",
            hail_size_cm=1.5,
        ),
    ),
    (
        "Negative 4 – Parked car + Clear weather (no event)",
        make_vehicle(
            vehicle_id="KL-07-QR-2468", speed_kmh=0, status="parked",
            location={"lat": 9.9312, "lon": 76.2673, "address": "Marine Drive, Kochi"},
        ),
        make_weather_event(
            event_type="clear", severity="none",
            distance_km=0, eta_minutes=0,
            description="Sunny skies, no weather hazards",
        ),
    ),
    (
        "Negative 5 – Moving car + Clear weather (double negative)",
        make_vehicle(
            vehicle_id="RJ-14-ST-3579", speed_kmh=85, status="moving",
            location={"lat": 26.9124, "lon": 75.7873, "address": "C-Scheme, Jaipur"},
        ),
        make_weather_event(
            event_type="clear", severity="none",
            distance_km=0, eta_minutes=0,
            description="Clear conditions throughout the region",
        ),
    ),
]

all_scenarios = positive_scenarios + negative_scenarios
print(f"✅ {len(positive_scenarios)} positive + {len(negative_scenarios)} negative scenarios ready.")

---
## Cell 7 — Run All Scenarios

Iterates through all 10 scenarios, evaluates each one, and prints the result.  
A summary at the end shows how many alerts were triggered vs suppressed.

In [ ]:
print("\n" + "#" * 70)
print("#   SAFE PILOT APP – Scenario 1: Weather Alerts for Parked Cars   #")
print("#" * 70)

triggered_count  = 0
suppressed_count = 0

for scenario_name, vehicle, weather in all_scenarios:
    result = evaluate_and_alert(vehicle, weather)
    print_result(scenario_name, vehicle, weather, result)
    if result["alert_triggered"]:
        triggered_count += 1
    else:
        suppressed_count += 1

print(f"\n{'=' * 70}")
print(f"  SUMMARY: {triggered_count} alert(s) triggered | {suppressed_count} suppressed")
print(f"{'=' * 70}\n")

---
## Cell 8 — Try Your Own Input (Optional)

Edit the values below and run the cell to test any custom vehicle + weather combination.

In [ ]:
# ── Customize vehicle state ──────────────────────────────────────────────────
my_vehicle = make_vehicle(
    vehicle_id = "MY-00-XX-0000",
    speed_kmh  = 0,               # 0 = parked; >0 = moving
    status     = "parked",        # "parked" | "idle" | "moving"
    location   = {"lat": 0.0, "lon": 0.0, "address": "My Location"},
)

# ── Customize approaching weather ────────────────────────────────────────────
my_weather = make_weather_event(
    event_type    = "hail",       # hail | thunderstorm | tornado | flash_flood
                                  # blizzard | high_winds | dust_storm | clear
    severity      = "severe",     # none | moderate | severe | extreme
    distance_km   = 12,           # km from vehicle
    eta_minutes   = 20,           # minutes until arrival
    description   = "Large hail approaching my area",
    hail_size_cm  = 2.5,          # optional — set None if not applicable
    rainfall_mm_hr= None,         # optional
    wind_speed_kmh= None,         # optional
)

# ── Evaluate and display ─────────────────────────────────────────────────────
my_result = evaluate_and_alert(my_vehicle, my_weather)
print_result("Custom Scenario", my_vehicle, my_weather, my_result)